## Domo AI Pro Operations & Processing
Domo AI Pro operations and processing will consume credits as described on [Domo's online consumption terms](https://www.domo.com/consumption-terms). Please see the credit rate card in your Domo instance for more information. (Admin > Company Settings > Credit Utilization > Rate Card)

# Messages API with domojupyter AI

This notebook demonstrates how to use `ai.messages()` to interact with a language model using
a structured message list — giving you direct control over the full conversation history.

`ai.messages()` differs from `ai.chat_completion()` in that you construct the entire message
list yourself, including prior user and assistant turns, rather than passing the current input
and history separately.

**Function signature:**
```python
ai.messages(
    messages,               # required: list of {"role": "user"|"assistant", "content": str} dicts
    system=None,            # optional: system message string
    model=None,             # optional: model ID
    temperature=None,       # optional: float
    max_tokens=None,        # optional: int
    reasoning_config=None,  # optional: dict e.g. {"enabled": True, "budgetTokens": 1000}
    response_format=None    # optional: dict e.g. {"type": "JSON", "schema": {...}}
)
# Returns: MessagesAIResponse with .text
```

In [ ]:
import domojupyter.ai as ai
import json

## 1. Single Message

Pass a list with a single user message to make a straightforward request.
This is the minimal usage of `ai.messages()`.

In [ ]:
response = ai.messages(
    messages=[
        {
            "role": "user",
            "content": (
                "Explain the difference between a leading indicator and a lagging indicator "
                "in the context of a sales pipeline dashboard."
            )
        }
    ]
)

print(response.text)

## 2. Multi-Turn Conversation

Build a full conversation history by alternating `user` and `assistant` messages in the list.
This gives you precise control over what prior context the model sees, which is useful for
constructing targeted analytical dialogues or replaying conversations from stored logs.

In [ ]:
response = ai.messages(
    messages=[
        {
            "role": "user",
            "content": "What metrics should I track to understand the health of a B2B customer account?"
        },
        {
            "role": "assistant",
            "content": (
                "For B2B account health, focus on: product usage frequency and depth, support ticket volume "
                "and severity trends, NPS or CSAT scores, contract renewal date and upsell history, "
                "engagement with customer success (QBR attendance, email open rates), and payment behavior "
                "such as days-to-pay and disputes. Together these form a composite health score."
            )
        },
        {
            "role": "user",
            "content": "Which of those are the strongest early warning signs of churn?"
        },
        {
            "role": "assistant",
            "content": (
                "The strongest early warning signs are: a sudden drop in product usage (especially among "
                "power users), a spike in support escalations, missed QBRs or unresponsive champions, "
                "and a declining NPS score. Usage decline is typically the earliest signal — often appearing "
                "60–90 days before a churn decision is made."
            )
        },
        {
            "role": "user",
            "content": (
                "If I have 90 days of daily active user data per account in a Domo dataset, "
                "what calculation would give me a meaningful usage trend signal?"
            )
        }
    ]
)

print(response.text)

## 3. With a System Message

Add a `system` parameter to set the model's role, tone, and behavioral constraints before
processing the messages list. The system message applies to the entire conversation.

In [ ]:
response = ai.messages(
    messages=[
        {
            "role": "user",
            "content": "We are evaluating whether to expand into the EMEA market next fiscal year. What financial thresholds should we meet first?"
        },
        {
            "role": "assistant",
            "content": (
                "Before expanding to EMEA, ensure you have: at least 18 months of operating runway, "
                "a domestic NRR above 110% (proving repeatable retention), at least 5 inbound EMEA leads "
                "per month without active marketing (demand signal), gross margin above 65% to absorb "
                "localization costs, and a payback period under 18 months in your current market."
            )
        },
        {
            "role": "user",
            "content": "We meet all of those except gross margin — we're at 61%. Should that be a blocker?"
        }
    ],
    system=(
        "You are a strategic finance advisor for high-growth SaaS companies. "
        "Be direct, quantitative where possible, and flag when assumptions matter. "
        "Do not hedge excessively — give a clear recommendation."
    )
)

print(response.text)

## 4. With Extended Thinking (reasoning_config)

Enable `reasoning_config` with a generous `budgetTokens` value to allow the model to reason
through complex, multi-variable problems before producing its answer. This is especially useful
for strategic analysis, trade-off evaluation, or decision-making scenarios where surface-level
pattern matching is insufficient.

In [ ]:
response = ai.messages(
    messages=[
        {
            "role": "user",
            "content": (
                "We are a Series B SaaS company with the following situation:\n"
                "- ARR: $18M, growing 35% YoY\n"
                "- Gross margin: 68%\n"
                "- Burn rate: $900K/month\n"
                "- Cash on hand: $22M\n"
                "- NRR: 108%\n"
                "- CAC payback: 22 months\n"
                "- Sales headcount: 12 AEs, 4 SDRs\n"
                "- Top 3 customers = 41% of ARR\n\n"
                "We have two options: (A) raise a Series C now at a $90M pre-money valuation "
                "to accelerate go-to-market, or (B) cut burn to $600K/month and reach "
                "cash-flow break-even in 18 months without raising. "
                "Analyze both options across the dimensions of risk, growth trajectory, "
                "dilution, and strategic positioning, and give a recommendation with reasoning."
            )
        }
    ],
    system=(
        "You are a venture-backed startup advisor with deep expertise in SaaS unit economics "
        "and fundraising strategy. Think carefully through the trade-offs before responding."
    ),
    reasoning_config={"enabled": True, "budgetTokens": 4000}
)

print(response.text)